In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from pymargins import GComputation  # 0.4.0: Margins -> GComputation

rng = np.random.default_rng(42)
n = 2000
df = pd.DataFrame({
    "age": rng.integers(20, 75, n),
    "treatment": rng.binomial(1, 0.40, n),
    "dose": rng.choice([0, 50, 100], n),
    "policy": rng.choice(["A", "B"], n),
})
lp = (-1.5 + 0.04 * df["age"] + 0.8 * df["treatment"]
      + 0.01 * df["dose"]
      + 0.3 * (df["policy"] == "B"))
df["y"] = rng.binomial(1, 1 / (1 + np.exp(-lp)))

fit = smf.glm("y ~ age + treatment + dose + C(policy)", data=df,
              family=sm.families.Binomial()).fit()
m = GComputation(fit, at="overall", scale="identity")

In [2]:
from pymargins import GComputation  # 0.4.0: Margins -> GComputation

m = GComputation(fit, at="overall", scale="identity")

scenarios = [
    {"atexog": {"treatment": 1}, "label": "treated"},
    {"atexog": {"treatment": 0}, "label": "control"},
]

res = m.evaluate(
    scenarios=scenarios,
    compose=lambda p: 1.0 / (p[0] - p[1]),
)
print(res.summary())

              Graph Result (delta, level=0.95)             
         estimate  std err       z  P>|z|  [95% Conf. Int.]
-----------------------------------------------------------
treated    8.8186   1.3457  6.5532  0.000   6.1811, 11.4561

n = 2000
Delta-vs-sim disagreement: 17.010%
plan a7666ac@1 | κ = 0.323


In [3]:
m = GComputation(fit, at="overall", scale="identity")

scenarios = [
    {"atexog": {"treatment": 1}, "label": "treated"},
    {"atexog": {"treatment": 0}, "label": "control"},
]

res = m.evaluate(
    scenarios=scenarios,
    compose=lambda p: p[0] / p[1],
)
print(res.summary())

              Graph Result (delta, level=0.95)              
         estimate  std err        z  P>|z|  [95% Conf. Int.]
------------------------------------------------------------
treated    1.1549   0.0255  45.2943  0.000    1.1049, 1.2048

n = 2000
Delta-vs-sim disagreement: 0.264%
plan a7666ac@1 | κ = 0.049


In [4]:
scenarios = [
    {"atexog": {"dose": 0}, "label": "placebo"},
    {"atexog": {"dose": 50}, "label": "low"},
    {"atexog": {"dose": 100}, "label": "high"},
]

# Emax-style parameter: (high − placebo) / (low − placebo)
res = m.evaluate(
    scenarios=scenarios,
    compose=lambda p: (p[2] - p[0]) / (p[1] - p[0]),
)

In [5]:
import jax.numpy as jnp

m = GComputation(fit, at="overall", scale="identity")

scenarios = [
    {"atexog": {"policy": "A"}, "label": "regime_A"},
    {"atexog": {"policy": "B"}, "label": "regime_B"},
]

res = m.evaluate(
    scenarios=scenarios,
    compose=lambda p: jnp.sqrt(p[0]) - jnp.sqrt(p[1]),
)
print(res.summary())

               Graph Result (delta, level=0.95)              
          estimate  std err        z  P>|z|  [95% Conf. Int.]
-------------------------------------------------------------
regime_A   -0.0289   0.0100  -2.8761  0.004  -0.0486, -0.0092

n = 2000
Delta-vs-sim disagreement: 1.160%
plan a7666ac@1 | κ = 0.035
